# Ordenar y subir fotos del dataset · Grupo 3 · Limón sutil

Cada integrante trabaja **solo en su carpeta** de Drive (`Ark`, `Franco`, `Guinno` o `Leoncio`), con sus tres clases
(`verde_…`, `pinton_…`, `maduro_…`). El notebook revisa qué hay, toma **el último número de cada clase** y continúa desde ahí.

Nombre de cada foto: **`CLASE_CELULAR_NÚMERO.jpg`** → `v_A26_604.jpg`, `p_RN10_001.jpg`, `m_R14P_201.jpg`.

| Integrante | Celular | Rango |
|---|---|---|
| Guinno | RN10 (Xiaomi Redmi Note 10 Pro) | 001–200 |
| Franco | R14P (Xiaomi Redmi 14 Pro) | 201–400 |
| Leoncio | A56 (Samsung Galaxy A56 5G) | 401–600 |
| Ark (Joel) | A26 (Samsung Galaxy A26) | 601–800 |

**Dos modos:**
- **Ordenar y renombrar lo que ya está en mi carpeta:** pone nombre a las fotos sin nombre y corrige nombres repetidos o mal escritos. No toca las que ya están bien.
- **Subir fotos nuevas:** eliges la clase, subes las fotos y quedan en tu carpeta con su nombre, después del último número.

La celda 4 **primero muestra la vista previa** (no cambia nada). Si está bien, haces clic en el botón rojo **APLICAR**. Cada cambio se **agrega al final** de un único archivo `registro_<Integrante>.csv` en tu carpeta.

## 1. Conectar con Drive

In [ ]:
# 1. Conectar con Google Drive y datos del equipo (se ejecuta una sola vez)
import os, io, re, hashlib, datetime
import pandas as pd
from PIL import Image
from googleapiclient.http import MediaIoBaseUpload

# carpeta del integrante: código del celular, primer número de su rango, texto esperado en el modelo del EXIF
# y códigos provisionales antiguos (las fotos con esos códigos solo cambian el código y conservan su número)
INTEGRANTES = {
    "Guinno":  dict(codigo="RN10", inicio=1,   modelo="Redmi Note 10 Pro", antiguos=[]),          # Xiaomi Redmi Note 10 Pro
    "Franco":  dict(codigo="R14P", inicio=201, modelo="",                  antiguos=["FRANCO"]),  # Xiaomi Redmi 14 Pro
    "Leoncio": dict(codigo="A56",  inicio=401, modelo="A56",               antiguos=["LEONCIO"]), # Samsung Galaxy A56 5G
    "Ark":     dict(codigo="A26",  inicio=601, modelo="A26",               antiguos=[]),          # Samsung Galaxy A26
}
TAM_RANGO = 200
LETRA = {"verde": "v", "pinton": "p", "maduro": "m"}
PATRON = re.compile(r"^([vpm])_([A-Za-z0-9]+)_(\d{3})\.jpg$")      # ej. m_RN10_001.jpg

if "drive" not in globals():
    from google.colab import auth
    from googleapiclient.discovery import build
    auth.authenticate_user()                                    # pide permiso a tu cuenta de Google
    drive = build("drive", "v3")
print("Conectado a Drive.")

## 2. Quién eres y qué quieres hacer

In [ ]:
# 2. Quién eres y qué quieres hacer
#@markdown Enlace de la carpeta **etiquetado_final_agregarfotos** (la que tiene Ark, Franco, Guinno y Leoncio).
CARPETA = "https://drive.google.com/drive/folders/1KqT60QOlIsdWQMDNuu0GVYzsaB4-WwLy"  #@param {type:"string"}
QUIEN = "Ark"  #@param ["Ark", "Franco", "Guinno", "Leoncio"]
MODO = "Ordenar y renombrar lo que ya está en mi carpeta"  #@param ["Ordenar y renombrar lo que ya está en mi carpeta", "Subir fotos nuevas"]
#@markdown Solo para **Subir fotos nuevas**:
CLASE = "verde"  #@param ["verde", "pinton", "maduro"]

m = re.search(r"folders/([A-Za-z0-9_-]+)", CARPETA)
RAIZ = m.group(1) if m else CARPETA.strip()
YO = INTEGRANTES[QUIEN]
FIN = YO["inicio"] + TAM_RANGO - 1
print(f"{QUIEN} · celular {YO['codigo']} · rango {YO['inicio']:03d}-{FIN:03d} · {MODO}")

## 3. Analizar mi carpeta

In [ ]:
# 3. Analizar mi carpeta: qué fotos hay, cuáles ya tienen nombre y cuál es el último número de cada clase
CAMPOS = "nextPageToken, files(id, name, mimeType, md5Checksum, size, createdTime, owners(emailAddress), imageMediaMetadata(time, cameraModel, width, height))"
def listar(padre, carpetas=False):
    q = f"'{padre}' in parents and trashed = false and mimeType " + ("=" if carpetas else "!=") + " 'application/vnd.google-apps.folder'"
    res, token = [], None
    while True:
        r = drive.files().list(q=q, fields=CAMPOS, pageSize=1000, pageToken=token,
                               supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
        res += r.get("files", []); token = r.get("nextPageToken")
        if not token: return res

# mi carpeta (Ark, Franco, Guinno o Leoncio) y sus carpetas de clase (verde_ark, pinton_ark, ... o verde, pinton, ...)
mias = [f for f in listar(RAIZ, carpetas=True) if f["name"].lower() == QUIEN.lower()]
assert mias, f"No encuentro la carpeta {QUIEN} dentro del enlace"
MI_CARPETA = mias[0]["id"]
SUB = {}
for f in listar(MI_CARPETA, carpetas=True):
    for clase in LETRA:
        if f["name"].lower().startswith(clase): SUB[clase] = f["id"]
print("Carpetas de clase encontradas:", list(SUB) or "ninguna")

filas = []
for clase, fid in SUB.items():
    for f in listar(fid):
        m = PATRON.match(f["name"]); meta = f.get("imageMediaMetadata", {})
        num = int(m.group(3)) if m else None
        en_rango = bool(m) and m.group(1) == LETRA[clase] and YO["inicio"] <= num <= FIN
        bien = en_rango and m.group(2) == YO["codigo"]
        recod = en_rango and m.group(2) in YO["antiguos"]              # código provisional antiguo: solo cambia el código
        filas.append(dict(clase=clase, id=f["id"], nombre=f["name"], bien=bien, numero=num if bien else None,
                          num_viejo=num if recod else None,
                          md5=f.get("md5Checksum"), kb=round(int(f.get("size", 0)) / 1024), mime=f.get("mimeType", ""),
                          fecha=str(meta.get("time") or f.get("createdTime", "")), celular=meta.get("cameraModel", ""),
                          ancho=meta.get("width", 0), alto=meta.get("height", 0),
                          dueño=(f.get("owners") or [{}])[0].get("emailAddress", "")))
D = pd.DataFrame(filas, columns=["clase", "id", "nombre", "bien", "numero", "num_viejo", "md5", "kb", "mime", "fecha", "celular",
                                 "ancho", "alto", "dueño"]).astype({"bien": bool})

# nombre repetido dentro de la misma clase: se queda la primera (por fecha), las demás se renumeran
D = D.sort_values(["clase", "fecha", "nombre"]).reset_index(drop=True)
D["repetido"] = D.bien & D.duplicated(["clase", "nombre"], keep="first")
D.loc[D.repetido, "bien"] = False; D.loc[D.repetido, "numero"] = None

def siguiente(clase):
    d = D[D.clase == clase]
    usados = pd.concat([d[d.bien].numero, d.num_viejo]).dropna()          # números con código bueno o antiguo
    return int(usados.max()) + 1 if len(usados) else YO["inicio"]

res = []
for clase in LETRA:
    d = D[D.clase == clase]
    res.append({"Clase": clase, "Fotos": len(d), "Con nombre correcto": int(d.bien.sum()),
                "Por renombrar": int((~d.bien).sum()), "Con código antiguo": int(d.num_viejo.notna().sum()), "Último número": int(d[d.bien].numero.max()) if d.bien.any() else "-",
                "Siguiente": siguiente(clase)})
print(f"\nCarpeta de {QUIEN}:"); print(pd.DataFrame(res).to_string(index=False))
if D.repetido.any(): print("\nNombres repetidos (se renumeran):", D[D.repetido].nombre.tolist())
dup = D[D.md5.notna() & D.md5.duplicated(keep=False)]
if len(dup): print("\nOJO, misma foto subida dos veces:"); print(dup.sort_values("md5")[["clase", "nombre"]].to_string(index=False))
otros = D[~D.dueño.str.contains("|".join(["joelitoea356lol", "fm71579489", "guinno.12", "lhuacacolquic", "upao.edu.pe"]), regex=True) & (D.dueño != "")]
if len(otros): print("\nFotos subidas por una cuenta que no es del grupo:", otros[["clase", "nombre", "dueño"]].values.tolist())

## 4. Vista previa y botón APLICAR

In [ ]:
# 4. Vista previa de los nombres nuevos y botón para aplicar
def avisos(kb, w, h, cel):
    a = []
    if kb and kb < 300: a.append("archivo muy liviano (¿WhatsApp?)")
    if (w, h) == (1056, 1056): a.append("1056x1056: posible foto de otra especie")
    if w and min(w, h) < 1000: a.append(f"resolución baja ({w}x{h})")
    if cel and "iphone" in cel.lower(): a.append(f"foto de iPhone ({cel}): no es el celular del integrante")
    elif YO["modelo"] and cel and YO["modelo"].lower() not in cel.lower(): a.append(f"celular distinto: {cel}")
    elif not YO["modelo"] and cel: a.append(f"celular según la foto: {cel}")
    return "; ".join(a)

plan = []
if MODO.startswith("Ordenar"):
    for clase in LETRA:
        n = siguiente(clase)
        ocupados = set(D[(D.clase == clase) & D.bien].numero)
        for r in D[(D.clase == clase) & ~D.bien].itertuples():            # ya ordenadas por fecha de toma
            if "iphone" in str(r.celular).lower():                          # celular que no es del integrante
                plan.append(dict(clase=clase, actual=r.nombre, nuevo="(se omite: foto de iPhone, moverla a descartadas)", id=r.id))
                continue
            if pd.notna(r.num_viejo) and int(r.num_viejo) not in ocupados:  # código antiguo: conserva su número
                num = int(r.num_viejo); ocupados.add(num)
            else:
                num = n; n += 1
            plan.append(dict(clase=clase, actual=r.nombre, nuevo=f"{LETRA[clase]}_{YO['codigo']}_{num:03d}.jpg", id=r.id,
                             avisos=avisos(r.kb, r.ancho, r.alto, r.celular) + ("" if "jpeg" in r.mime else "; no es JPG")))
else:
    from google.colab import files
    subido = files.upload()                                       # elegir las fotos nuevas de esa clase
    n = siguiente(CLASE); ya = set(D.md5.dropna())
    def fecha_local(datos):
        try:
            ex = Image.open(io.BytesIO(datos)).getexif(); return str(ex.get_ifd(0x8769).get(36867) or ex.get(306) or "")
        except Exception: return ""
    for nombre, datos in sorted(subido.items(), key=lambda kv: (fecha_local(kv[1]), kv[0])):
        md5 = hashlib.md5(datos).hexdigest()
        if md5 in ya: plan.append(dict(clase=CLASE, actual=nombre, nuevo="(repetida: ya está en Drive)")); continue
        try: im = Image.open(io.BytesIO(datos)); w, h = im.size; cel = str(im.getexif().get(272) or "")
        except Exception: plan.append(dict(clase=CLASE, actual=nombre, nuevo="(no es una imagen)")); continue
        if "iphone" in cel.lower(): plan.append(dict(clase=CLASE, actual=nombre, nuevo="(se omite: foto de iPhone)")); continue
        if im.format != "JPEG":                                   # PNG/HEIC/WEBP -> JPG
            b = io.BytesIO(); im.convert("RGB").save(b, "JPEG", quality=95, exif=im.getexif()); datos = b.getvalue()
        plan.append(dict(clase=CLASE, actual=nombre, nuevo=f"{LETRA[CLASE]}_{YO['codigo']}_{n:03d}.jpg", datos=datos,
                         avisos=avisos(len(datos) / 1024, w, h, cel)))
        ya.add(md5); n += 1

P = pd.DataFrame(plan, columns=["clase", "actual", "nuevo", "id", "datos", "avisos"])
validos = P[P.nuevo.str.endswith(".jpg")]
assert all(int(x[-7:-4]) <= FIN for x in validos.nuevo), f"Se pasa del rango de {QUIEN} ({FIN})"
if not len(P):
    print("Todo en orden: no hay nada que renombrar.")
else:
    print(P[["clase", "actual", "nuevo", "avisos"]].fillna("").to_string(index=False))
    print(f"\nVISTA PREVIA: todavía no se cambió nada ({len(validos)} fotos).")

    def aplicar():
        for r in validos.itertuples():
            if MODO.startswith("Ordenar"):
                drive.files().update(fileId=r.id, body={"name": r.nuevo}, supportsAllDrives=True).execute()
            else:
                if CLASE not in SUB:                               # crea la carpeta de la clase si no existe
                    SUB[CLASE] = drive.files().create(body={"name": f"{CLASE}_{QUIEN.lower()}", "parents": [MI_CARPETA],
                                 "mimeType": "application/vnd.google-apps.folder"}, supportsAllDrives=True).execute()["id"]
                drive.files().create(body={"name": r.nuevo, "parents": [SUB[CLASE]]},
                                     media_body=MediaIoBaseUpload(io.BytesIO(r.datos), mimetype="image/jpeg"),
                                     supportsAllDrives=True).execute()
        print(f"Listo: {len(validos)} fotos {'renombradas' if MODO.startswith('Ordenar') else 'subidas'}.")
        # registro único por integrante: se agregan las filas nuevas al final de registro_<QUIEN>.csv
        reg = P.drop(columns=["datos", "id"]).assign(integrante=QUIEN, fecha_cambio=f"{datetime.datetime.now():%Y-%m-%d %H:%M}")
        nombre_reg = f"registro_{QUIEN}.csv"
        leer = lambda f: pd.read_csv(io.BytesIO(drive.files().get_media(fileId=f["id"], supportsAllDrives=True).execute()),
                                     encoding="utf-8-sig")
        previos = sorted((f for f in listar(MI_CARPETA) if f["name"].startswith(f"registro_{QUIEN}") and f["name"].endswith(".csv")),
                         key=lambda f: f["name"])
        unico = [f for f in previos if f["name"] == nombre_reg]
        base = [leer(unico[0])] if unico else [leer(f) for f in previos]     # la primera vez junta los registros antiguos
        todo = pd.concat(base + [reg], ignore_index=True)
        media = MediaIoBaseUpload(io.BytesIO(todo.to_csv(index=False).encode("utf-8-sig")), mimetype="text/csv")
        if unico:
            drive.files().update(fileId=unico[0]["id"], media_body=media, supportsAllDrives=True).execute()
        else:
            drive.files().create(body={"name": nombre_reg, "parents": [MI_CARPETA]}, media_body=media, supportsAllDrives=True).execute()
        print(f"Registro actualizado: {nombre_reg} ({len(reg)} filas nuevas, {len(todo)} en total). Ejecuta la celda 3 para ver cómo quedó.")

    if len(validos):
        import ipywidgets as widgets
        from IPython.display import display
        boton = widgets.Button(description=f"APLICAR ({len(validos)} fotos)", button_style="danger",
                               icon="check", layout=widgets.Layout(width="260px", height="40px"))
        salida = widgets.Output()
        def al_click(b):
            b.disabled = True; b.description = "Aplicando..."          # evita doble clic
            with salida:
                try:
                    aplicar(); b.description = "Hecho ✔"; b.button_style = "success"
                except Exception as e:
                    print("Error:", e); b.description = "Error (ver abajo)"
        boton.on_click(al_click)
        print("Si la lista está bien, haz clic en el botón:")
        display(boton, salida)

### Notas
- Quien lo ejecute debe ser **editor** de la carpeta.
- Las fotos no se modifican: renombrar solo cambia el nombre; al subir, los JPG se guardan tal cual.
- Si una foto está en la clase equivocada, muévela a la carpeta correcta en Drive y vuelve a ejecutar en modo **Ordenar**: le pone la letra correcta.
- Códigos definitivos (21-oct): Franco **R14P**, Leoncio **A56**. Las fotos con el código provisional (`FRANCO`, `LEONCIO`) se corrigen en modo **Ordenar**: solo cambia el código y **conservan su número**.
- Cada integrante toma sus fotos con **su propio celular**. Si la vista previa avisa "foto de iPhone" o "celular distinto", esa foto no va.
- Fotos que no se usan: muévelas a una subcarpeta como `descartadas_<integrante>` (el notebook solo mira las carpetas verde, pinton y maduro).
- El registro es un solo archivo por integrante (`registro_<Integrante>.csv`). La primera vez junta los registros antiguos con fecha; esos se pueden borrar después.